<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_250_Inferencia_sobre_medias_pruebas_t_independientes_y_relacionadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 250. Inferencia sobre medias: pruebas t para muestras independientes y relacionadas</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 5. Generalización de conclusiones**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Retos en la app:** [las tres misiones del Tema 250](https://fborrasumh.github.io/bioestadistica/#tema-250)

**Evaluación:** el examen del Módulo 5 se hace **con Colab**. Practica ejecutando e interpretando cada análisis.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** antes de empezar, usa **Entorno de ejecución → Ejecutar todo**. Así se dibujan los gráficos. Las celdas con borde azul son las diapositivas. Debajo de cada una está el desarrollo del bloque; pliégalo con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Elegir entre la prueba *t* para una muestra, para dos muestras independientes y para muestras relacionadas.
2. Aplicar la prueba *t* de Welch y saber por qué es la opción recomendada para dos grupos independientes.
3. Estimar la diferencia de medias con su intervalo de confianza y el tamaño del efecto (*d* de Cohen).
4. Comprobar las condiciones de aplicación de forma gráfica y razonada.
5. Reconocer cuándo conviene una alternativa no paramétrica.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: las tres pruebas t | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">HDL en hombres y mujeres de la cohorte</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Dos grupos distintos de pacientes: prueba t de Welch.</li><li>Se comunica la diferencia de medias con su IC, el valor p y el tamaño del efecto.</li></ul></div>

In [ ]:
#@title Figura: HDL en hombres y mujeres de la cohorte
import pandas as pd
import matplotlib.pyplot as plt

datos = pd.read_csv("https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/cohorte_limpia.csv").dropna(subset=["hdl_mg_dl"])
grupos = [datos.loc[datos["sexo"] == s, "hdl_mg_dl"] for s in ["Hombre", "Mujer"]]
plt.figure(figsize=(9, 4.5))
plt.boxplot(grupos, widths=0.5, patch_artist=True, boxprops=dict(facecolor="lightsteelblue"), medianprops=dict(color="firebrick", lw=2))
for i, g in enumerate(grupos, start=1):
    plt.scatter([i] * len(g) + (pd.Series(range(len(g))) % 7 - 3) * 0.03, g, s=8, alpha=0.4, color="steelblue")
    plt.text(i + 0.3, g.mean(), f"media {g.mean():.1f}", fontsize=12, va="center")
plt.xticks([1, 2], ["Hombres", "Mujeres"])
plt.ylabel("HDL (mg/dL)")
plt.title("¿Diferencia real o azar del muestreo? ¿De qué tamaño?", fontweight="bold")
plt.show()

#### Desarrollo para el estudio

En la cohorte cardiovascular, el colesterol HDL medio es de unos 49 mg/dL en los hombres y de unos 58 mg/dL en las mujeres. Antes de proponer puntos de corte distintos por sexo en el protocolo del centro, el equipo quiere saber dos cosas. ¿Es real esa diferencia en la población? ¿**De qué tamaño** es?

Además, una enfermera ha medido la presión arterial a 30 hipertensos **antes y después** de 8 semanas de un programa de reducción de sal. Quiere saber si la PAS ha bajado.

Las dos preguntas comparan **medias**, pero la segunda usa los **mismos** pacientes dos veces. Cada una requiere una versión distinta de la prueba *t* de Student.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Las tres pruebas t</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Tres pruebas t para tres diseños</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_250_tres.png" width="780" alt="Tres pruebas t para tres diseños"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Una media frente a un valor de referencia · dos grupos independientes · los mismos pacientes dos veces.</li><li>El diseño, no los datos, decide la prueba.</li></ul></div>

#### Desarrollo para el estudio

| Prueba | Pregunta | Hipótesis nula | Ejemplo |
| --- | --- | --- | --- |
| *t* para una muestra | ¿La media es distinta de un valor de referencia? | μ = μ₀ | ¿La HbA1c media de nuestros diabéticos difiere del objetivo del 7 %? |
| *t* para dos muestras independientes | ¿Difieren las medias de dos grupos distintos? | μ₁ = μ₂ | HDL en hombres y mujeres |
| *t* para muestras relacionadas (apareadas) | ¿Cambia la media en los mismos individuos? | μ de las diferencias = 0 | PAS antes y después del programa |

En todas, el estadístico tiene la misma estructura:

$$t = \frac{\text{diferencia observada} - \text{diferencia bajo } H_0}{\text{error estándar de la diferencia}}$$

### <font color='steelblue'>2.2 Welch o Student clásica</font>

La prueba *t* clásica para dos muestras supone que las **varianzas son iguales** en los dos grupos. La versión de **Welch** no lo supone y ajusta los grados de libertad. Cuando las varianzas son iguales, las dos dan casi lo mismo; cuando no lo son, la clásica puede equivocarse. Por eso se recomienda usar **Welch por defecto**, sin contrastar antes la igualdad de varianzas.

### <font color='steelblue'>2.3 Muestras relacionadas</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Antes y después en los mismos pacientes</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_250_apareados.png" width="780" alt="Antes y después en los mismos pacientes"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>La variabilidad entre pacientes es mucho mayor que la del cambio dentro de cada paciente.</li><li>La prueba apareada analiza las diferencias: elimina esa variabilidad y gana potencia.</li></ul></div>

#### Desarrollo para el estudio

Con datos apareados se calcula para cada paciente la **diferencia** (después − antes) y se aplica la *t* para una muestra a esas diferencias. La prueba apareada elimina la variabilidad entre pacientes. Por eso es mucho más potente que comparar los dos momentos como grupos independientes (igual que McNemar frente a la *z* en el Tema 240).

### <font color='steelblue'>2.4 Condiciones de aplicación</font>

<img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_250_esquema.png" width="760" alt="Tres comprobaciones antes de la prueba t: independencia, normalidad o n ≥ 30 y ausencia de atípicos extremos, con la alternativa de cada una.">

| Condición | Cómo valorarla | Si no se cumple |
| --- | --- | --- |
| Observaciones independientes (dentro de cada grupo) | Por el diseño | Otro método (datos agrupados, medidas repetidas) |
| Normalidad de la variable (o de las diferencias) | Histograma y gráfico Q-Q; con *n* ≥ 30 por grupo la prueba es robusta por el teorema central del límite | Transformación (logaritmo) o prueba no paramétrica |
| Ausencia de atípicos extremos | Diagrama de caja | Revisar si son errores (Tema 100); prueba no paramétrica |

Los contrastes formales de normalidad (Shapiro-Wilk) tienen poco valor: con muestras pequeñas casi nunca detectan nada, y con muestras grandes detectan desviaciones irrelevantes.

### <font color='steelblue'>2.5 Tamaño del efecto: d de Cohen</font>

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{conjunta}}}$$

Expresa la diferencia en unidades de desviación típica. Orientación habitual: 0,2 pequeño, 0,5 mediano, 0,8 grande. Para decidir la relevancia clínica, sin embargo, importa más la diferencia en las **unidades originales** (mg/dL, mmHg) comparada con la mínima diferencia clínicamente importante.

## <font color='steelblue'>3. Demostración: las tres pruebas t (20 min)</font>

### <font color='steelblue'>3.1 Dos muestras independientes: HDL por sexo</font>

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

cohorte = pd.read_csv("https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/cohorte_limpia.csv")
hdl = cohorte.dropna(subset=["hdl_mg_dl"])
hombres = hdl.loc[hdl["sexo"] == "Hombre", "hdl_mg_dl"]
mujeres = hdl.loc[hdl["sexo"] == "Mujer", "hdl_mg_dl"]

print(hdl.groupby("sexo")["hdl_mg_dl"].agg(["count", "mean", "std"]).round(1))

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=hdl, x="sexo", y="hdl_mg_dl", ax=ejes[0])
ejes[0].set_ylabel("HDL (mg/dL)")
for grupo, color in [(hombres, "steelblue"), (mujeres, "indianred")]:
    stats.probplot(grupo, plot=ejes[1])
ejes[1].set_title("Gráfico Q-Q de los dos grupos")
plt.tight_layout()
plt.show()

Los diagramas de caja muestran distribuciones razonablemente simétricas y los puntos del gráfico Q-Q siguen aproximadamente una recta. Con más de 170 pacientes por grupo, la prueba *t* es además robusta a desviaciones moderadas de la normalidad.

In [ ]:
welch = stats.ttest_ind(mujeres, hombres, equal_var=False)
clasica = stats.ttest_ind(mujeres, hombres, equal_var=True)
ic = welch.confidence_interval()

n1, n2 = len(mujeres), len(hombres)
s_conjunta = np.sqrt(((n1 - 1) * mujeres.var() + (n2 - 1) * hombres.var()) / (n1 + n2 - 2))
d = (mujeres.mean() - hombres.mean()) / s_conjunta

print(f"Diferencia (mujeres − hombres): {mujeres.mean() - hombres.mean():.1f} mg/dL "
      f"(IC 95 %: {ic.low:.1f} a {ic.high:.1f})")
print(f"Welch:   t = {welch.statistic:.2f}, gl = {welch.df:.0f}, p = {welch.pvalue:.1e}")
print(f"Clásica: t = {clasica.statistic:.2f}, gl = {clasica.df:.0f}, p = {clasica.pvalue:.1e}")
print(f"d de Cohen = {d:.2f}")

**Cómo se comunica:** «El HDL medio fue de 58,4 mg/dL (DE 12,5) en mujeres y de 48,8 mg/dL (DE 9,7) en hombres. Diferencia de 9,6 mg/dL (IC 95 %: 7,3 a 11,9; prueba *t* de Welch, *p* < 0,001; *d* = 0,86)».

Una diferencia de casi 10 mg/dL es grande y clínicamente relevante. Por eso las guías usan umbrales de HDL bajo distintos por sexo (< 40 mg/dL en hombres y < 50 mg/dL en mujeres).

### <font color='steelblue'>3.2 Muestras relacionadas: el programa de reducción de sal</font>

In [ ]:
rng = np.random.default_rng(seed=250)
antes = rng.normal(148, 14, 30).round()
despues = (antes - rng.normal(5, 6, 30)).round()
diferencias = despues - antes

apareada = stats.ttest_rel(despues, antes)
ic = apareada.confidence_interval()
print(f"PAS media antes: {antes.mean():.1f} · después: {despues.mean():.1f} mmHg")
print(f"Cambio medio: {diferencias.mean():.1f} mmHg (IC 95 %: {ic.low:.1f} a {ic.high:.1f})")
print(f"t apareada: t = {apareada.statistic:.2f}, gl = {apareada.df:.0f}, p = {apareada.pvalue:.1e}")

uno = stats.ttest_1samp(diferencias, 0)
print(f"t para una muestra sobre las diferencias: t = {uno.statistic:.2f}, p = {uno.pvalue:.1e} (idéntica)")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for a, d_ in zip(antes, despues):
    ejes[0].plot([0, 1], [a, d_], color="gray", alpha=0.5)
ejes[0].set_xticks([0, 1], ["Antes", "Después"])
ejes[0].set_ylabel("PAS (mmHg)")
ejes[0].set_title("Cada línea es un paciente")
ejes[1].hist(diferencias, bins=10, color="lightsteelblue", edgecolor="white")
ejes[1].axvline(0, color="black", linestyle="--")
ejes[1].set_xlabel("Cambio de PAS (mmHg)")
ejes[1].set_title("Distribución de las diferencias")
plt.tight_layout()
plt.show()

### <font color='steelblue'>3.3 El error de ignorar el apareamiento</font>

In [ ]:
indep = stats.ttest_ind(despues, antes, equal_var=False)
print(f"Como si fueran grupos independientes: p = {indep.pvalue:.3f}")
print(f"Prueba apareada (correcta):           p = {apareada.pvalue:.1e}")
print(f"DE de la PAS entre pacientes: {antes.std(ddof=1):.1f} · DE de las diferencias: {diferencias.std(ddof=1):.1f} mmHg")

La variabilidad **entre pacientes** (DE de unos 13 mmHg) es mucho mayor que la variabilidad del **cambio** dentro de cada paciente. La prueba apareada la elimina; la independiente la deja como ruido y no detecta el efecto.

Como en el Tema 240: sin grupo control no se puede atribuir la reducción al programa. Parte puede deberse a la regresión a la media, porque se eligieron pacientes con la PAS alta.

### <font color='steelblue'>3.4 Una muestra: HbA1c de los diabéticos frente al objetivo</font>

In [ ]:
analiticas = pd.read_csv("https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/analiticas.csv")
datos = cohorte.merge(analiticas, on="id")
hba1c = datos.loc[datos["diabetes"] == "Sí", "hba1c_pct"].dropna()

r = stats.ttest_1samp(hba1c, popmean=7.0)
ic = r.confidence_interval()
print(f"Diabéticos con HbA1c: {len(hba1c)}")
print(f"HbA1c media: {hba1c.mean():.2f} % (IC 95 %: {ic.low:.2f} a {ic.high:.2f})")
print(f"Frente al objetivo del 7 %: t = {r.statistic:.2f}, p = {r.pvalue:.3f}")
print(f"Pacientes por encima del objetivo: {(hba1c > 7).mean():.0%}")

La media está por encima del objetivo. Pero el dato clínicamente útil es también **qué proporción de pacientes** no lo alcanza. Una media cercana al objetivo puede esconder a muchos pacientes mal controlados.

### <font color='steelblue'>Ampliación: cuando la normalidad falla</font>

Con muestras pequeñas y variables muy asimétricas (como la creatinina o el NT-proBNP), hay dos opciones:

- **Transformar** la variable con el logaritmo y aplicar la *t* a los logaritmos. La diferencia de medias de los logaritmos, exponenciada, es un **cociente de medias geométricas**.
- Usar una prueba **no paramétrica**: Mann-Whitney para dos grupos independientes y Wilcoxon de los rangos con signo para datos apareados (Tema 260).

In [ ]:
creatinina = datos.dropna(subset=["creatinina_mg_dl"])
diab = creatinina.loc[creatinina["diabetes"] == "Sí", "creatinina_mg_dl"]
no_diab = creatinina.loc[creatinina["diabetes"] == "No", "creatinina_mg_dl"]

r_log = stats.ttest_ind(np.log(diab), np.log(no_diab), equal_var=False)
ic = r_log.confidence_interval()
cociente = np.exp(np.log(diab).mean() - np.log(no_diab).mean())
print(f"Cociente de medias geométricas (diabéticos / no diabéticos): {cociente:.2f} "
      f"(IC 95 %: {np.exp(ic.low):.2f} a {np.exp(ic.high):.2f}), p = {r_log.pvalue:.3f}")
print(f"Mann-Whitney: p = {stats.mannwhitneyu(diab, no_diab).pvalue:.3f}")

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Usar la *t* para muestras independientes con datos apareados.**

**Contrastar la igualdad de varianzas para elegir la prueba.** Usa Welch directamente.

**Exigir un Shapiro-Wilk no significativo** con muestras grandes: la *t* es robusta.

**Dar solo el valor p.** Da las medias, la diferencia, su IC y el tamaño del efecto.

**Hacer muchas pruebas *t* entre varios grupos** en lugar de un análisis de la varianza (Tema 260).

### <font color='steelblue'>Ampliación: Comparar las varianzas de dos grupos</font>

La *t* clásica supone que los dos grupos tienen la **misma varianza**. Hay dos pruebas para comprobarlo:

- **Prueba F:** compara el cociente de las dos varianzas. Exige normalidad.
- **Prueba de Levene:** es más robusta si los datos no son normales.

En ambas, H₀ dice que las varianzas son iguales. Muchos autores recomiendan usar **Welch siempre**: funciona bien con varianzas iguales y con varianzas distintas. Así la decisión no depende de otra prueba previa.

**En tus materiales SIMUSTAT:** [Tema 400. Contraste de dos medias poblacionales](https://colab.research.google.com/drive/1URRViPWC_adC8BGr6Kn_Z8fjV4UGjgcA), apartado 2.

In [ ]:
f = mujeres.var() / hombres.var()
gl1, gl2 = len(mujeres) - 1, len(hombres) - 1
p_f = 2 * min(stats.f.sf(f, gl1, gl2), stats.f.cdf(f, gl1, gl2))
lev = stats.levene(mujeres, hombres)

print(f"DE del HDL: mujeres {mujeres.std():.1f} | hombres {hombres.std():.1f} mg/dL")
print(f"Prueba F:   F = {f:.2f}, p = {p_f:.3f}")
print(f"Levene:     W = {lev.statistic:.2f}, p = {lev.pvalue:.3f}")

Si las varianzas difieren, la *t* clásica puede dar un valor p erróneo. Welch, en cambio, sigue siendo válida. Usa la comparación de varianzas para describir los grupos, no para elegir la prueba.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 250 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** elegir la prueba *t* adecuada para seis diseños y calcular a mano un estadístico *t*.
2. **Misión clínica (con pistas):** comparar el IMC entre diabéticos y no diabéticos de la cohorte. Dar la diferencia, el IC 95 %, la *d* de Cohen y una frase de resultados.
3. **Misión reto (con asistente):** pedir al asistente el análisis de un ensayo antes-después, detectar si ha usado la prueba equivocada y corregirla.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'Se mide la glucemia a 25 pacientes antes y después de un tratamiento. ¿Qué prueba '
              'usas?',
  'type': 'multiple_choice',
  'answers': [{'answer': 't para muestras relacionadas',
               'correct': True,
               'feedback': 'Correcto: son los mismos pacientes medidos dos veces.'},
              {'answer': 't de Welch para muestras independientes',
               'correct': False,
               'feedback': 'Ignoraría el apareamiento y perdería potencia.'},
              {'answer': 'Ji cuadrado',
               'correct': False,
               'feedback': 'La variable es cuantitativa.'},
              {'answer': 'McNemar',
               'correct': False,
               'feedback': 'McNemar es para variables binarias apareadas.'}]},
 {'question': '¿Por qué se recomienda la prueba t de Welch para dos grupos independientes?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Porque no supone varianzas iguales y funciona bien tanto si lo son como '
                         'si no',
               'correct': True,
               'feedback': 'Correcto: evita contrastar antes la igualdad de varianzas.'},
              {'answer': 'Porque no supone normalidad',
               'correct': False,
               'feedback': 'Welch también se basa en la distribución t.'},
              {'answer': 'Porque sirve para datos apareados',
               'correct': False,
               'feedback': 'Es para muestras independientes.'},
              {'answer': 'Porque siempre da valores p más pequeños',
               'correct': False,
               'feedback': 'No es así.'}]},
 {'question': 'Una diferencia de medias de PAS es de 4 mmHg (IC 95 %: −1 a 9). ¿Qué concluyes?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'No es estadísticamente significativa; el efecto real podría ser nulo o '
                         'de hasta 9 mmHg',
               'correct': True,
               'feedback': 'Correcto: el IC contiene el 0.'},
              {'answer': 'La diferencia es de 4 mmHg con seguridad',
               'correct': False,
               'feedback': '4 mmHg es la estimación puntual.'},
              {'answer': 'El tratamiento empeora la PAS',
               'correct': False,
               'feedback': 'El IC incluye valores en los dos sentidos.'},
              {'answer': 'Es significativa porque 4 > 0',
               'correct': False,
               'feedback': 'La significación depende del IC.'}]},
 {'question': '¿Qué expresa una d de Cohen de 0,8?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Una diferencia de medias de 0,8 desviaciones típicas, considerada grande',
               'correct': True,
               'feedback': 'Correcto.'},
              {'answer': 'Una diferencia de 0,8 en las unidades de la variable',
               'correct': False,
               'feedback': 'La d está estandarizada.'},
              {'answer': 'Una probabilidad del 80 % de que la diferencia sea real',
               'correct': False,
               'feedback': 'No es una probabilidad.'},
              {'answer': 'Una correlación de 0,8',
               'correct': False,
               'feedback': 'No es una correlación.'}]},
 {'question': 'Con 12 pacientes por grupo y una variable muy asimétrica con atípicos, ¿qué es lo '
              'más adecuado?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Transformar con logaritmos o usar la prueba de Mann-Whitney',
               'correct': True,
               'feedback': 'Correcto: con muestras pequeñas la t no es robusta a esa asimetría.'},
              {'answer': 'La t clásica, porque n es pequeño',
               'correct': False,
               'feedback': 'Con n pequeño la normalidad importa más.'},
              {'answer': 'Eliminar los atípicos y hacer la t',
               'correct': False,
               'feedback': 'Solo se eliminan si son errores comprobados.'},
              {'answer': 'La t apareada',
               'correct': False,
               'feedback': 'Los grupos son independientes.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- *t* para una muestra, para dos independientes (Welch) o para relacionadas, según el diseño.
- La *t* apareada es la *t* para una muestra aplicada a las diferencias.
- Da siempre la diferencia de medias con su IC y el tamaño del efecto.
- Con *n* moderado la *t* es robusta; con *n* pequeño y asimetría, logaritmos o pruebas no paramétricas.
- Antes-después sin grupo control no demuestra causalidad.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Prueba *t* para una muestra | Contraste de una media frente a un valor de referencia |
| Prueba *t* para dos muestras independientes | Contraste de la igualdad de las medias de dos grupos distintos |
| Prueba *t* de Welch | Versión de la *t* para dos muestras que no supone varianzas iguales |
| Prueba *t* para muestras relacionadas | Contraste de la media de las diferencias apareadas frente a 0 |
| Gráfico Q-Q | Gráfico que compara los cuantiles de los datos con los de una distribución normal |
| *d* de Cohen | Diferencia de medias dividida por la desviación típica conjunta |
| Media geométrica | Exponencial de la media de los logaritmos |
| Prueba de Mann-Whitney | Alternativa no paramétrica a la *t* para dos muestras independientes |

## <font color='steelblue'>Para saber más</font>

- Ruxton GD. The unequal variance t-test is an underused alternative to Student's t-test and the Mann-Whitney U test. *Behav Ecol*. 2006;17(4):688-690.
- Bland JM, Altman DG. Transformations, means, and confidence intervals. *BMJ*. 1996;312(7038):1079.

## <font color='steelblue'>Retos de este tema en la app</font>

Practica lo que has visto con las tres misiones del Tema 250 en la app **Bioestadística**. Cuentan para la evaluación continua.

1. **Básica (sin ayuda):** Elegir la prueba t
2. **Clínica (con pistas):** IMC de diabéticos y no diabéticos
3. **Reto (con asistente de IA):** Un ensayo antes-después: ¿la prueba equivocada?

<a href="https://fborrasumh.github.io/bioestadistica/#tema-250" target="_blank" style="display:inline-block;padding:10px 18px;background:#C8461F;color:#fff;border-radius:8px;font-weight:bold;text-decoration:none">Abrir los retos del Tema 250 en la app ↗</a>

---
*Tema 250 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*